# Marker quality across all P7–P11 trials

For every `Trial{1,2}_{handsonly,hoi}` recording of `P7`, `P8`, `P9`, `P10`,
and `P11`:

1. Load the raw Nexus export (`vicon2mano.loader.load_csv`). Actual filenames
   aren't consistent across participants (typos, underscore variants, mixed
   case), so `find_trial_files` matches each CSV to a canonical trial key —
   first via `trial_filename_map.csv` (an explicit, hand-editable
   participant→filename→trial_key table you can correct), then by keyword
   guess (trial number + "hoi"/"hand"/"only"/"static" in the name) for any
   file not listed there.
2. If a participant has a `static`-classified recording, load it too and use
   *all* its frames as extra bone-length reference data (see step 3) — a
   calibration/static trial is expected to have every marker present and
   stable throughout, which is far richer than the sparse spot-checked
   frames alone. Its markers are realigned to each trial's own column order
   by label name (`align_markers_to_labels`), since it's a separate
   recording. **Caveat, seen on real data**: this is only valid if the
   static recording shares the same rigid marker geometry as the trials —
   `P9`'s static file, `static_after_forerm1_fall.csv`, was recorded after a
   forearm marker fell off / was reattached, so its forearm-plate geometry
   doesn't match the trials; including it collapsed 3 of P9's 4 trials to 0%
   correct. It's now excluded via `trial_filename_map.csv` (`trial_key` left
   blank for that row). Check the printed "using ... as an extra
   bone-length reference" line and the per-trial mean before trusting any
   *other* static-augmented result the same way.
3. Look up the manually-verified good frames from `manual_frames.csv`
   (`scripts/label_marker_quality.load_ref_ranges_csv`) — each number in a
   row is an *individually* spot-checked good frame, not a start/end range.
   Trials with no row yet fall back to `auto_reference` (the longest
   run(s) of frames where every marker is present) — unverified, flagged as
   such in the printed summary.
4. Label every marker in every frame `correct` / `incorrect` / `missing`
   with a **cascading trust-chain check**
   (`scripts/label_marker_quality.label_quality_cascade`):
   1. A frame is only usable as an anchor if all 4 Forearm markers are
      present and the rigid Forearm plate's own pairwise distances are
      within tolerance (this has nothing to do with hand pose — the plate
      is rigid regardless of gesture).
   2. In anchor-trustworthy frames, each Palm marker and each finger's
      *first* marker is checked against its reference distance to the
      forearm centroid; in frames where the anchor can't be trusted, these
      are conservatively marked incorrect rather than assumed correct.
   3. Every other finger marker is checked against the *preceding* marker
      on the same finger (consecutive bone length), same as before.
   4. A temporal (frame-to-frame speed) check on top, same as before.
   5. **Forearm veto**: if the anchor isn't trustworthy in a frame, every
      present marker in that frame is forced to `incorrect`, overriding
      whatever the earlier stages computed for it.
5. Compute the per-frame percentage of correctly-labelled markers.
6. Animate a trial with each marker circled green (correct) or red
   (incorrect); missing markers are omitted.

Caveat: trials with a `manual_frames.csv` row currently have only a handful
of spot-checked good frames (typically 3), usually non-consecutive. That's
enough for the bone-length and forearm-anchor checks (a per-bone median/MAD
from those frames, supplemented by any static recording) but too sparse for
a per-marker *speed* reference, so the temporal check falls back to a fixed
mm/frame ceiling instead of an adaptive one. Add more spot-checked frames to
`manual_frames.csv` — ideally some consecutive runs — to tighten this.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

REPO_ROOT = Path.cwd()
for p in (REPO_ROOT, REPO_ROOT / "scripts"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from vicon2mano.loader import load_csv
import label_marker_quality as lmq

## Configuration

In [ ]:
DATA_ROOT = Path(r"D:\ExperimentsJune25")
REF_CSV = DATA_ROOT / "manual_frames.csv"
TRIAL_MAP_CSV = DATA_ROOT / "trial_filename_map.csv"
PARTICIPANTS = ["P7",] # "P8", "P9", "P10", "P11"]
TRIALS = ["Trial1_handsonly", "Trial1_hoi", "Trial2_handsonly", "Trial2_hoi"]


def load_trial_filename_map(path: Path) -> dict[tuple[str, str], str]:
    """Read trial_filename_map.csv: participant,filename,trial_key.

    An explicit, hand-editable override for find_trial_files' keyword
    guess — e.g. a filename the "1"/"2" + "hoi"/"hand"/"only"/"static"
    heuristic gets wrong. Blank trial_key means "not a trial at all"
    (excluded — a recording that isn't a real trial or a usable static
    reference, e.g. an aborted take).
    """
    import csv as csv_mod

    mapping: dict[tuple[str, str], str] = {}
    if not path.exists():
        return mapping
    with path.open(newline="", encoding="utf-8-sig") as f:
        for row in csv_mod.DictReader(f):
            participant = (row.get("participant") or "").strip()
            filename = (row.get("filename") or "").strip()
            trial_key = (row.get("trial_key") or "").strip()
            if participant and filename:
                mapping[(participant, filename.lower())] = trial_key
    return mapping


def find_trial_files(participant: str, participant_dir: Path,
                      overrides: dict[tuple[str, str], str] | None = None) -> dict[str, Path]:
    """Map canonical trial keys ("Trial1_handsonly", ..., and "static" for a
    calibration/static recording if present) to actual CSV files in a
    participant's directory.

    Filenames aren't consistent across participants (typos like
    "Trail1_HOI", underscore variants like "Trial1_hands_only", mixed
    case). ``overrides`` (from trial_filename_map.csv) is consulted first
    for an explicit assignment; any file not listed there falls back to a
    keyword guess (trial number "1"/"2" + "hoi"/"hand"/"only" in the name,
    or "static" in the name for a calibration recording).
    """
    overrides = overrides or {}
    found: dict[str, Path] = {}
    for csv_path in sorted(participant_dir.glob("*.csv")):
        key = overrides.get((participant, csv_path.name.lower()))
        if key is None:
            # Not in the map at all (a new/unseen file) — fall back to guessing.
            name = csv_path.stem.lower()
            if "static" in name:
                key = "static"
            elif "1" in name or "2" in name:
                trial_num = "1" if "1" in name else "2"
                if "hoi" in name:
                    kind = "hoi"
                elif "hand" in name or "only" in name:
                    kind = "handsonly"
                else:
                    continue
                key = f"Trial{trial_num}_{kind}"
            else:
                continue
        elif key == "":
            continue  # explicitly marked "not a trial" in the map
        if key in found:
            print(f"[warn] {participant_dir}: both {found[key].name} and {csv_path.name} "
                  f"match {key!r} — keeping {found[key].name}")
            continue
        found[key] = csv_path
    return found


def auto_reference(markers, min_frames=300):
    """Fallback when a trial has no row in manual_frames.csv yet: the
    longest run(s) of frames where every marker is present. Unverified --
    prefer adding a real reference range to manual_frames.csv instead."""
    present = np.isfinite(markers).all(axis=(1, 2))
    runs, start = [], None
    for t, ok in enumerate(present):
        if ok and start is None:
            start = t
        elif not ok and start is not None:
            runs.append((start, t)); start = None
    if start is not None:
        runs.append((start, len(present)))
    runs.sort(key=lambda r: r[1] - r[0], reverse=True)
    idx = []
    for s, e in runs:
        idx.extend(range(s, e))
        if len(idx) >= min_frames:
            break
    return np.array(sorted(idx), dtype=int)

## Load + label every trial

Each trial's result is kept in `results[(participant, trial)]` as
`(markers, labels, bones, status, pct_correct)`.

In [ ]:
results = {}
trial_map = load_trial_filename_map(TRIAL_MAP_CSV)

for participant in PARTICIPANTS:
    participant_dir = DATA_ROOT / participant / participant
    trial_files = find_trial_files(participant, participant_dir, trial_map)

    static_path = trial_files.get("static")
    static_markers_raw, static_labels = (None, None)
    if static_path is not None:
        static_markers_raw, static_labels = load_csv(str(static_path))
        print(f"{participant}: using {static_path.name} ({static_markers_raw.shape[0]} frames) "
              f"as an extra bone-length reference")

    for trial in TRIALS:
        path = trial_files.get(trial)
        if path is None:
            print(f"[skip] {participant}/{trial}: no matching CSV in {participant_dir}")
            continue

        markers, labels = load_csv(str(path))

        static_markers = None
        if static_markers_raw is not None:
            static_markers = lmq.align_markers_to_labels(static_markers_raw, static_labels, labels)

        try:
            ref_frames = lmq.load_ref_ranges_csv(str(REF_CSV), participant, trial)
            ref_source = "manual_frames.csv"
        except ValueError:
            ref_frames = auto_reference(markers)
            ref_source = "AUTO (unverified — add a row to manual_frames.csv)"
        ref_frames = ref_frames[(ref_frames >= 0) & (ref_frames < markers.shape[0])]

        status, bones = lmq.label_quality_cascade(markers, labels, ref_frames, static_markers=static_markers)
        pct_correct = (status == 2).sum(axis=1) / status.shape[1] * 100

        results[(participant, trial)] = (markers, labels, bones, status, pct_correct)
        print(f"{participant}/{trial}: {markers.shape[0]} frames, "
              f"ref={ref_source} ({len(ref_frames)} frames), "
              f"mean correct {pct_correct.mean():0.1f}% "
              f"(min {pct_correct.min():.0f}%, max {pct_correct.max():.0f}%)")

## Summary: % correct over time, all trials

In [ ]:
fig, axes = plt.subplots(len(PARTICIPANTS), len(TRIALS), figsize=(4 * len(TRIALS), 3 * len(PARTICIPANTS)),
                          sharey=True, squeeze=False)
for row, participant in enumerate(PARTICIPANTS):
    for col, trial in enumerate(TRIALS):
        ax = axes[row, col]
        key = (participant, trial)
        if key not in results:
            ax.axis("off")
            continue
        _, _, _, _, pct_correct = results[key]
        ax.plot(pct_correct, color="#2980b9", linewidth=0.7)
        ax.set_title(f"{participant}/{trial}\nmean {pct_correct.mean():.0f}%", fontsize=9)
        ax.set_ylim(0, 100)
        if col == 0:
            ax.set_ylabel("% correct")
fig.tight_layout()
plt.show()

## Animate one trial (interactive 3D)

Plotly `Scatter3d`, so you can drag to orbit, scroll to zoom, and scrub the
slider or hit Play — no re-running cells needed to change the view.

Each marker's **fill color is its finger** (thumb/index/middle/ring/pinky/
palm/wrist/forearm) and its **outline is the quality check**: green =
correct, red = incorrect; missing markers are omitted entirely. Hover a
marker to see its label. The title and slider show the frame index and the
percentage of correctly-labelled markers that frame. Gray lines are the
inferred skeleton bones.

Change `ANIMATE_TRIAL` to any key printed above; `N_OUT` controls how many
evenly-sampled frames the timelapse covers.

In [ ]:
import plotly.graph_objects as go

ANIMATE_TRIAL = ("P7", "Trial1_hoi")
N_OUT = 300

# Original marker colors, by finger (same palette as the static skeleton plot).
FINGER_PALETTE = {
    "thumb": "#e74c3c", "index": "#27ae60", "middle": "#2980b9",
    "ring": "#e67e22", "pinky": "#8e44ad", "palm": "#444444",
    "wrist": "#444444", "forearm": "#95a5a6",
}
STATUS_OUTLINE = {0: "rgba(0,0,0,0)", 1: "#e74c3c", 2: "#2ecc71"}  # missing, incorrect, correct


def marker_color(label):
    low = label.lower()
    for key, color in FINGER_PALETTE.items():
        if key in low:
            return color
    return "#000000"


def marker_trace_data(markers, status, t):
    fm = markers[t]
    present = np.isfinite(fm).all(axis=-1)
    x = np.where(present, fm[:, 0], np.nan)
    y = np.where(present, fm[:, 1], np.nan)
    z = np.where(present, fm[:, 2], np.nan)
    outline = [STATUS_OUTLINE[int(s)] for s in status[t]]
    return x, y, z, outline


def bone_lines(frame_markers, bones):
    """(x, y, z) with None gaps so all bone segments render as one trace."""
    xs, ys, zs = [], [], []
    for i, j, _ in bones:
        if np.isfinite(frame_markers[i]).all() and np.isfinite(frame_markers[j]).all():
            xs += [frame_markers[i, 0], frame_markers[j, 0], None]
            ys += [frame_markers[i, 1], frame_markers[j, 1], None]
            zs += [frame_markers[i, 2], frame_markers[j, 2], None]
    return xs, ys, zs

In [ ]:
markers, labels, bones, status, pct_correct = results[ANIMATE_TRIAL]
participant, trial = ANIMATE_TRIAL

frame_idx = np.linspace(0, markers.shape[0] - 1, N_OUT).astype(int)
center = np.nanmean(markers, axis=(0, 1))
half = 200  # mm
fill_colors = [marker_color(l) for l in labels]


def frame_title(t):
    return f"{participant}/{trial}  frame {t}  correct {pct_correct[t]:.0f}%"


x0, y0, z0, outline0 = marker_trace_data(markers, status, frame_idx[0])
bx0, by0, bz0 = bone_lines(markers[frame_idx[0]], bones)

# Plotly's Scatter3d marker outline (`line=`) renders as a hairline in
# WebGL regardless of `width` — draw the status ring as a separate, larger
# marker layer behind the fill layer instead, which is reliably visible.
ring_trace = go.Scatter3d(
    x=x0, y=y0, z=z0, mode="markers",
    marker=dict(size=14, color=outline0), hoverinfo="skip", name="status",
)
marker_trace = go.Scatter3d(
    x=x0, y=y0, z=z0, mode="markers",
    marker=dict(size=6, color=fill_colors),
    text=labels, hoverinfo="text", name="markers",
)
bone_trace = go.Scatter3d(
    x=bx0, y=by0, z=bz0, mode="lines",
    line=dict(color="lightgray", width=2), hoverinfo="skip", name="bones",
)

# Legend proxy traces: no real data, just a swatch + label per finger group
# and per quality status, since Scatter3d doesn't expose a discrete-color
# legend the way px does.
finger_legend_traces = [
    go.Scatter3d(
        x=[None], y=[None], z=[None], mode="markers",
        marker=dict(size=6, color=color), name=finger,
        legendgroup="finger", legendgrouptitle=dict(text="Finger"),
    )
    for finger, color in FINGER_PALETTE.items()
]
status_legend_traces = [
    go.Scatter3d(
        x=[None], y=[None], z=[None], mode="markers",
        marker=dict(size=10, color=color), name=label,
        legendgroup="status", legendgrouptitle=dict(text="Status"),
    )
    for label, color in [("correct", STATUS_OUTLINE[2]), ("incorrect", STATUS_OUTLINE[1])]
]

frames = []
for k, t in enumerate(frame_idx):
    x, y, z, outline = marker_trace_data(markers, status, t)
    bx, by, bz = bone_lines(markers[t], bones)
    frames.append(go.Frame(
        name=str(k),
        data=[
            go.Scatter3d(x=x, y=y, z=z, marker=dict(color=outline)),
            go.Scatter3d(x=x, y=y, z=z, marker=dict(color=fill_colors)),
            go.Scatter3d(x=bx, y=by, z=bz),
        ],
        layout=go.Layout(title=frame_title(t)),
    ))

fig = go.Figure(
    data=[ring_trace, marker_trace, bone_trace, *finger_legend_traces, *status_legend_traces],
    frames=frames,
    layout=go.Layout(
        title=frame_title(frame_idx[0]),
        width=950, height=800,
        legend=dict(x=1.02, y=1),
        scene=dict(
            xaxis=dict(range=[center[0] - half, center[0] + half], title="X (mm)"),
            yaxis=dict(range=[center[1] - half, center[1] + half], title="Y (mm)"),
            zaxis=dict(range=[center[2] - half, center[2] + half], title="Z (mm)"),
            aspectmode="cube",
        ),
        updatemenus=[dict(
            type="buttons", showactive=False,
            buttons=[
                dict(label="Play", method="animate", args=[
                    None, {"frame": {"duration": 60, "redraw": True},
                           "fromcurrent": True, "transition": {"duration": 0}}]),
                dict(label="Pause", method="animate", args=[
                    [None], {"frame": {"duration": 0}, "mode": "immediate"}]),
            ],
        )],
        sliders=[dict(
            currentvalue=dict(prefix="frame: "),
            steps=[
                dict(method="animate", label=str(t),
                     args=[[str(k)], {"frame": {"duration": 0, "redraw": True}, "mode": "immediate"}])
                for k, t in enumerate(frame_idx)
            ],
        )],
    ),
)

# include_plotlyjs=True embeds the library inline instead of fetching it from
# a CDN — the CDN silently renders a blank div if this machine has no (or
# restricted) internet access, with no visible error.
from IPython.display import HTML
HTML(fig.to_html(include_plotlyjs=True, full_html=False))

## Debug a specific frame's cascade reasoning

`label_marker_quality.debug_frame_cascade` re-runs the exact same
stage-by-stage logic as `label_quality_cascade` for one frame and prints
every pairwise/anchor distance it checked, the reference it checked
against, and the pass/fail verdict at each step — including whether a
pair had *no reference at all* (silently skipped, which can hide a truly
bad marker). Use it whenever `results[...]`'s per-frame `status` looks
wrong for a marker you can see is off.

In [ ]:
DEBUG_TRIAL = ("P7", "Trial1_hoi")
DEBUG_FRAME = 17540

d_participant, d_trial = DEBUG_TRIAL
d_participant_dir = DATA_ROOT / d_participant / d_participant
d_trial_files = find_trial_files(d_participant, d_participant_dir, trial_map)

d_static_path = d_trial_files.get("static")
d_static_markers_raw = d_static_labels = None
if d_static_path is not None:
    d_static_markers_raw, d_static_labels = load_csv(str(d_static_path))

d_markers, d_labels = load_csv(str(d_trial_files[d_trial]))

d_static_markers = None
if d_static_markers_raw is not None:
    d_static_markers = lmq.align_markers_to_labels(d_static_markers_raw, d_static_labels, d_labels)

try:
    d_ref_frames = lmq.load_ref_ranges_csv(str(REF_CSV), d_participant, d_trial)
except ValueError:
    d_ref_frames = auto_reference(d_markers)
d_ref_frames = d_ref_frames[(d_ref_frames >= 0) & (d_ref_frames < d_markers.shape[0])]

lmq.debug_frame_cascade(d_markers, d_labels, d_ref_frames, DEBUG_FRAME, static_markers=d_static_markers)